# Patent inventors — `inventor_list`, `n_inventors`, first and last inventor

One row per US **utility** patent that has at least one disambiguated inventor, keyed on
`patent_id` like every other file in `PatentView/output/`. The PatentView twin of
`OpenAlex/output/paper_author.parquet` (`author_list, team_size, first_author, last_author`) and of
`Dimensions/output/paper_author.parquet`. `patent_inventor_country.parquet` is the geography
companion: it is built from the same rows and shares the `n_inventors` column with this file.

## Raw data
PatentsView **Granted** bulk `.tsv.zip` (2026-05-21 snapshot):
```
g_patent.tsv.zip                    patent_id, patent_type, patent_date    -> utility filter, grant_year
g_inventor_disambiguated.tsv.zip    patent_id, inventor_sequence, inventor_id
```
`inventor_id` is PatentsView's disambiguated inventor key: `fl:jo_ln:marron-5` for 98 % of the
rows, a 25-character base-36 hash (`0000n8nqsxhrztn7djlxou00k`) or a UUID for the rest.
`inventor_sequence` is the order in which the inventors are printed
on the grant, starting at 0. The file itself is **not** sorted by patent, so a row limit (smoke
test) truncates patents rather than taking whole ones.

## Two traps
1. **One inventor, several rows.** A disambiguated id can occur more than once on one patent when
   several raw name variants collapsed onto the same id. Everything below de-duplicates on
   `(patent_id, inventor_id)` first, keeping the lowest sequence. That is the rule behind
   `patent_metadata.inventor_list` and `patent_inventor_country.n_inventors` too, so the three
   files agree by construction — and the verification section asserts that they do.
2. **`;` inside an id.** A few dozen inventor ids carry a `;` of their own (an HTML entity such as
   `fl:iv_ln:jadri&cacute;-1`, or a mangled ß as in `fl:jo_ln:ha;feld-1`). `n_inventors` is a row
   count, not a split length, so the file is right; a consumer that does `inventor_list.split(';')`
   over-counts those patents. The check below splits where the **next id starts** instead, and
   reports how many patents a naive split would get wrong.

## Output — `PatentView/output/patent_inventor.parquet`
| column | |
|---|---|
| `patent_id` | utility patent with a grant date and ≥ 1 disambiguated inventor |
| `inventor_list` | distinct inventor ids in sequence order, `;`-joined |
| `n_inventors` | distinct inventors — `paper_author.team_size` under its PatentView name |
| `first_inventor` | id at the lowest `inventor_sequence` |
| `last_inventor` | id at the highest `inventor_sequence` (the same id on a single-inventor patent) |

Environment hooks for a smoke test: `NB_ROW_LIMIT` (read only the first N inventor rows) and
`NB_OUT_DIR` (write somewhere other than `PatentView/output`).

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv

OUT_DIR    = os.environ.get('NB_OUT_DIR', pv.OUT)
ROW_LIMIT  = int(os.environ['NB_ROW_LIMIT']) if os.environ.get('NB_ROW_LIMIT') else None
OUT_FP     = f'{OUT_DIR}/patent_inventor.parquet'
META_FP    = pv.out('patent_metadata.parquet')            # siblings, for the cross-checks only
COUNTRY_FP = pv.out('patent_inventor_country.parquet')
os.makedirs(OUT_DIR, exist_ok=True)
pv.preflight('patent_inventor')
print(f'pandas {pd.__version__} from {os.path.dirname(pd.__file__)}   duckdb {duckdb.__version__}')
if ROW_LIMIT:
    print(f'\n*** SMOKE TEST: first {ROW_LIMIT:,} inventor rows only -> {OUT_FP}')

# Never let pandas decide what a missing value looks like: an id could read 'NA', 'None' or 'null'.
READ = dict(sep='\t', dtype=str, keep_default_na=False, na_values=[''])

granted    : /project/jevans/Dawoon/Science of Science/PatentView/Granted   (35 zip files)
pregranted : /project/jevans/Dawoon/Science of Science/PatentView/Pregranted   (25 files)
output     : /project/jevans/Dawoon/Science of Science/PatentView/output

  patent_inventor                 OK
pandas 2.2.3 from /project/jevans/Dawoon/env/Curvature/lib/python3.12/site-packages/pandas   duckdb 1.5.5


## 1. Utility patent universe (same rule as `patent_metadata` and `patent_inventor_country`)

In [2]:
%%time
gp = pd.read_csv(pv.granted('g_patent.tsv.zip'), usecols=['patent_id', 'patent_type', 'patent_date'], **READ)
n_all = len(gp)
gp = gp[gp['patent_type'] == 'utility']
gp['grant_year'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
years = gp.dropna(subset=['grant_year']).set_index('patent_id')['grant_year'].astype(int)
util  = set(years.index)
print(f'{n_all:,} patents of all types -> {len(util):,} utility patents with a grant date')
del gp; gc.collect()

9,454,161 patents of all types -> 8,531,961 utility patents with a grant date


## 2. Inventor rows, one per (patent, inventor)

Rows are dropped in three named steps so the counts can be read off: not a utility patent, no
inventor id, and the same inventor listed twice on one patent. The sort key is
`(patent_id, inventor_sequence, inventor_id)`; the id tiebreak only matters if two rows of one
patent share a sequence, and makes the output deterministic if they do. `pos` is the dense
0..n-1 rank in that order, computed **in pandas** so the aggregation step can order by it and the
verification step can check the ordering by a different route.

In [3]:
%%time
t0 = time.time()
d = pd.read_csv(pv.granted('g_inventor_disambiguated.tsv.zip'),
                usecols=['patent_id', 'inventor_sequence', 'inventor_id'], nrows=ROW_LIMIT, **READ)
n_raw = len(d)
d = d[d['patent_id'].isin(util)];            n_util = len(d)
d = d.dropna(subset=['inventor_id']);        n_id   = len(d)
d['seq'] = pd.to_numeric(d['inventor_sequence'], errors='coerce')
n_null_seq = int(d['seq'].isna().sum())
d = d.sort_values(['patent_id', 'seq', 'inventor_id'], na_position='last')
n_pair_dup = int(d.duplicated(['patent_id', 'inventor_id']).sum())
d = d.drop_duplicates(['patent_id', 'inventor_id'], keep='first')
d['pos'] = d.groupby('patent_id', sort=False).cumcount().astype('int32')
inv = d[['patent_id', 'inventor_id', 'seq', 'pos']].reset_index(drop=True)
del d; gc.collect()

semi     = inv['inventor_id'].str.contains(';', regex=False)
shape_ok = inv['inventor_id'].str.match(r'^(fl:|[0-9a-z]{25}$|[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$)')
print(f'{n_raw:>12,} rows read'
      f'\n{n_raw - n_util:>12,} dropped: not a utility patent with a grant date'
      f'\n{n_util - n_id:>12,} dropped: empty inventor_id'
      f'\n{n_pair_dup:>12,} dropped: the same inventor listed twice on one patent (lowest sequence kept)'
      f'\n{len(inv):>12,} (patent, inventor) rows on {inv.patent_id.nunique():,} patents   [{time.time()-t0:.0f}s]'
      f'\n{n_null_seq:>12,} rows with a non-numeric inventor_sequence (placed after the numbered ones, ordered by id)'
      f'\n{int(semi.sum()):>12,} inventor ids containing ";"   e.g. {inv.inventor_id[semi].unique()[:4].tolist()}'
      f'\n{int((~shape_ok).sum()):>12,} inventor ids that are none of "fl:...", a 25-character hash, a UUID'
      f'   (the verification split rule assumes there are none)'
      f'\n{"":>12}  id shapes: fl: {int(inv.inventor_id.str.startswith("fl:").sum()):,}   '
      f'25-char hash {int(inv.inventor_id.str.fullmatch(r"[0-9a-z]{25}").sum()):,}   '
      f'UUID {int(inv.inventor_id.str.match(r"^[0-9a-f]{8}-").sum()):,}')

  24,037,380 rows read
   1,716,947 dropped: not a utility patent with a grant date
           0 dropped: empty inventor_id
       1,905 dropped: the same inventor listed twice on one patent (lowest sequence kept)
  22,318,528 (patent, inventor) rows on 8,531,216 patents   [251s]
           0 rows with a non-numeric inventor_sequence (placed after the numbered ones, ordered by id)
          33 inventor ids containing ";"   e.g. ['fl:br_ln:popovi&cacute;-1', 'fl:da_ln:fili&cacute;-1', 'fl:mi_ln:dumi&cacute;-1', 'fl:bo_ln:klepi&cacute;-1']
           0 inventor ids that are none of "fl:...", a 25-character hash, a UUID   (the verification split rule assumes there are none)
              id shapes: fl: 21,963,642   25-char hash 354,886   UUID 0


## 3. Aggregate per patent

`string_agg(... ORDER BY pos)` puts `inventor_list` in sequence order; `arg_min` / `arg_max` on
the same `pos` pick its ends through a different aggregate, so the check that they *are* the ends
of the list is not a tautology of one code path. Written sorted by `patent_id`, so a later join
against a sibling output scans a few row groups instead of the file.

In [4]:
%%time
con = duckdb.connect()
con.execute("SET preserve_insertion_order=false")
con.register('inv', inv)
t0 = time.time()
con.execute(f"""
COPY (
  SELECT patent_id,
         string_agg(inventor_id, ';' ORDER BY pos)   AS inventor_list,
         count(*)::INTEGER                            AS n_inventors,
         arg_min(inventor_id, pos)                    AS first_inventor,
         arg_max(inventor_id, pos)                    AS last_inventor
  FROM inv
  GROUP BY patent_id
  ORDER BY patent_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e6:.1f} MB, {n:,} patents) in {time.time()-t0:.0f}s')

WROTE /project/jevans/Dawoon/Science of Science/PatentView/output/patent_inventor.parquet  (292.2 MB, 8,531,216 patents) in 14s


## 4. Verification

1. `patent_id` unique; no nulls; `n_inventors` equals the length of `inventor_list`; the list holds
   no duplicate id; `first_inventor` / `last_inventor` are its two ends. The list is split where
   the **next id starts** (`;fl:`, `;` + a 25-character hash, or `;` + a UUID), not at every `;`,
   because a few ids contain a `;` of their own — `naive_split_overcounts` is how many patents a
   plain `split(';')` gets wrong.
2. Against `patent_metadata.inventor_list`: the **strings** must be identical on every shared
   patent (same source, same de-duplication rule, same order), and the two files must cover the
   same patents.
3. Against `patent_inventor_country.n_inventors`: identical counts on the same patents.
4. Team-size distribution, by grant decade, and the largest teams.

In a smoke test (`NB_ROW_LIMIT`) checks 2 and 3 are expected to disagree: the inventor file is not
sorted by patent, so a row limit leaves most patents with only some of their inventors.

In [5]:
%%time
def parts(col):
    # RE2 has no lookahead: mark each id start with '|' via a capture group, then split on it.
    # An id starts with 'fl:', or is a 25-char base-36 hash, or a UUID (8 hex, '-', 4 hex, '-').
    return f"str_split(regexp_replace({col}, ';(fl:|[0-9a-z]{{25}}|[0-9a-f]{{8}}-[0-9a-f]{{4}}-)', '|\\1', 'g'), '|')"

chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, {parts('inventor_list')} AS parts FROM s)
SELECT count(*)                                                                  AS patents,
       count(*) - count(DISTINCT patent_id)                                      AS dup_patent_id,
       count(*) FILTER (WHERE inventor_list IS NULL OR first_inventor IS NULL OR last_inventor IS NULL) AS nulls,
       sum(n_inventors)                                                          AS inventor_slots,
       count(*) FILTER (WHERE n_inventors <> len(parts))                         AS size_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))           AS dup_in_list,
       count(*) FILTER (WHERE parts[1] <> first_inventor)                        AS first_mismatch,
       count(*) FILTER (WHERE parts[len(parts)] <> last_inventor)                AS last_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(str_split(inventor_list, ';')))  AS naive_split_overcounts,
       count(*) FILTER (WHERE n_inventors = 1)                                   AS solo_patents,
       max(n_inventors)                                                          AS max_inventors
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<24} {int(v):>14,}')
ok = all(chk[k] == 0 for k in ['dup_patent_id', 'nulls', 'size_mismatch', 'dup_in_list', 'first_mismatch', 'last_mismatch'])
print(f"\n1.   {'ALL PASS' if ok else 'FAILED -- see the counts above'}"
      f"   (mean team size {chk.inventor_slots/chk.patents:.3f}, solo {chk.solo_patents/chk.patents*100:.1f}%)")

def report(step, label, n_bad, expected_in_smoke=True):
    tag = '   OK' if n_bad == 0 else ('   (expected in a smoke test: NB_ROW_LIMIT truncates patents)'
                                     if ROW_LIMIT and expected_in_smoke else '   <-- FAILED')
    print(f'{step}   {label}: {int(n_bad):,}{tag}')

if os.path.exists(META_FP):
    m = con.execute(f"""
    WITH n AS (SELECT patent_id, inventor_list FROM read_parquet('{OUT_FP}')),
         m AS (SELECT patent_id, inventor_list FROM read_parquet('{META_FP}') WHERE inventor_list IS NOT NULL)
    SELECT (SELECT count(*) FROM n JOIN m USING (patent_id))                                          AS shared,
           (SELECT count(*) FROM n JOIN m USING (patent_id) WHERE n.inventor_list <> m.inventor_list) AS list_mismatch,
           (SELECT count(*) FROM n ANTI JOIN m USING (patent_id))                                     AS only_here,
           (SELECT count(*) FROM m ANTI JOIN n USING (patent_id))                                     AS only_metadata
    """).fetchdf().iloc[0]
    print(f'\n2.   vs patent_metadata.inventor_list: {int(m.shared):,} shared patents')
    report('    ', 'inventor_list differs', m.list_mismatch)
    report('    ', 'patents here but without an inventor_list there', m.only_here, expected_in_smoke=False)
    report('    ', 'patents with an inventor_list there but absent here', m.only_metadata)
else:
    print('\n2.   patent_metadata.parquet not present -- cross-check skipped')

if os.path.exists(COUNTRY_FP):
    c = con.execute(f"""
    WITH n AS (SELECT patent_id, n_inventors FROM read_parquet('{OUT_FP}')),
         c AS (SELECT patent_id, n_inventors FROM read_parquet('{COUNTRY_FP}'))
    SELECT (SELECT count(*) FROM n JOIN c USING (patent_id))                                       AS shared,
           (SELECT count(*) FROM n JOIN c USING (patent_id) WHERE n.n_inventors <> c.n_inventors)  AS n_mismatch,
           (SELECT count(*) FROM n ANTI JOIN c USING (patent_id))                                  AS only_here,
           (SELECT count(*) FROM c ANTI JOIN n USING (patent_id))                                  AS only_country
    """).fetchdf().iloc[0]
    print(f'\n3.   vs patent_inventor_country.n_inventors: {int(c.shared):,} shared patents')
    report('    ', 'n_inventors differs', c.n_mismatch)
    report('    ', 'patents here but not there', c.only_here, expected_in_smoke=False)
    report('    ', 'patents there but not here', c.only_country)
else:
    print('\n3.   patent_inventor_country.parquet not present -- cross-check skipped')

print('\n4.   team-size distribution')
display(con.execute(f"""
SELECT CASE WHEN n_inventors >= 10 THEN '10+' ELSE lpad(n_inventors::VARCHAR, 2, ' ') END AS n_inventors,
       count(*) AS patents, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY 1""").fetchdf())
con.register('years', years.rename('grant_year').reset_index())
print('     by grant decade')
display(con.execute(f"""
SELECT (y.grant_year // 10) * 10 AS decade, count(*) AS patents,
       round(avg(s.n_inventors), 3) AS mean_inventors,
       round(100.0 * count(*) FILTER (WHERE s.n_inventors = 1) / count(*), 1)  AS pct_solo,
       round(100.0 * count(*) FILTER (WHERE s.n_inventors >= 5) / count(*), 1) AS pct_5plus
FROM read_parquet('{OUT_FP}') s JOIN years y USING (patent_id) GROUP BY 1 ORDER BY 1""").fetchdf())
print('     largest teams')
display(con.execute(f"""
SELECT patent_id, n_inventors, first_inventor, last_inventor, length(inventor_list) AS list_chars
FROM read_parquet('{OUT_FP}') ORDER BY n_inventors DESC, patent_id LIMIT 5""").fetchdf())
print('     first rows')
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') LIMIT 5").fetchdf())
con.close()

  patents                       8,531,216
  dup_patent_id                         0
  nulls                                 0
  inventor_slots               22,318,528
  size_mismatch                         0
  dup_in_list                           0
  first_mismatch                        0
  last_mismatch                         0
  naive_split_overcounts               25
  solo_patents                  2,936,874
  max_inventors                       133

1.   ALL PASS   (mean team size 2.616, solo 34.4%)

2.   vs patent_metadata.inventor_list: 8,531,216 shared patents
       inventor_list differs: 0   OK
       patents here but without an inventor_list there: 0   OK
       patents with an inventor_list there but absent here: 0   OK

3.   vs patent_inventor_country.n_inventors: 8,531,216 shared patents
       n_inventors differs: 0   OK
       patents here but not there: 0   OK
       patents there but not here: 0   OK

4.   team-size distribution


,n_inventors,patents,pct
0,1,2936874,34.43
1,2,2088166,24.48
2,3,1468645,17.21
3,4,916153,10.74
4,5,501284,5.88
5,6,271586,3.18
6,7,145185,1.70
7,8,81094,0.95
8,9,46080,0.54
9,10+,76149,0.89


     by grant decade


,decade,patents,mean_inventors,pct_solo,pct_5plus
0,1970,250255,1.715,57.7,2.7
1,1980,708173,1.856,52.8,4.0
2,1990,1108463,2.197,43.6,7.8
3,2000,1624898,2.473,36.7,11.0
4,2010,2866800,2.794,29.6,15.1
5,2020,1972627,3.098,24.9,19.6


     largest teams


,patent_id,n_inventors,first_inventor,last_inventor,list_chars
0,10367677,133,fl:st_ln:parkvall-1,fl:ya_ln:zheng-37,2523
1,11381445,133,fl:st_ln:parkvall-1,fl:ya_ln:zheng-35,2523
2,12231284,133,fl:st_ln:parkvall-1,fl:ya_ln:zheng-35,2523
3,10630410,130,fl:st_ln:parkvall-1,fl:ya_ln:zheng-35,2466
4,10638253,130,fl:st_ln:parkvall-1,fl:ya_ln:zheng-35,2466


     first rows


,patent_id,inventor_list,n_inventors,first_inventor,last_inventor
0,4442439,fl:ak_ln:mizuno-8,1,fl:ak_ln:mizuno-8,fl:ak_ln:mizuno-8
1,4442440,fl:gi_ln:elchinger-1,1,fl:gi_ln:elchinger-1,fl:gi_ln:elchinger-1
2,4442441,fl:yo_ln:kikuchi-29;fl:ta_ln:omori-18;fl:ha_ln...,5,fl:yo_ln:kikuchi-29,fl:to_ln:murayama-17
3,4442442,fl:ga_ln:o'dell-1,1,fl:ga_ln:o'dell-1,fl:ga_ln:o'dell-1
4,4442443,fl:jo_ln:martner-2,1,fl:jo_ln:martner-2,fl:jo_ln:martner-2


## 5. Reading the results

- Check 1 passing means the file is internally consistent under the *correct* split rule;
  `naive_split_overcounts` is the number of patents on which `inventor_list.split(';')` would
  return one item too many — use `n_inventors`, not the split length, for team size.
- Check 2 with zero mismatches and zero one-sided patents means this file and
  `patent_metadata.inventor_list` are the same information; this one adds `first_inventor` /
  `last_inventor` and can be rebuilt in minutes without the 868 MB metadata pass, in the same way
  `paper_author.parquet` sits beside `paper_metadata.parquet` on the paper side.
- Check 3 ties `n_inventors` to the geography file, so a join of the two on `patent_id` never has
  to reconcile two team sizes.